# Enrichissement historique du Rhône

Ce travail de Nouhayla porte sur 13 481 logements du Rhône. Il nécessite les fichiers `output/ademe_data_69_cleaned.csv` et `data/climate_zones.csv`, absents du poste audité. Il ne constitue pas l'entrée commune des modèles actuels. Le notebook 1 fournit déjà altitude et zones H1/H2/H3 dans `output/cleaned_data.csv`. Ne pas remplacer ces zones par H1c/H2d sans réentraîner les modèles. Les corrections des codes INSEE de Lyon et Pierre-Bénite méritent une intégration ultérieure, avec nouvelle version du CSV et des modèles.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pyproject.toml").exists() and (p / "MLmodels").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Ouvrez le notebook depuis le dépôt App_Enedis.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Racine du projet :", ROOT)

import pandas as pd
import numpy as np


In [ ]:
# Le périmètre Rhône ne doit pas être remplacé silencieusement par le CSV commun.
required_paths = [
    ROOT / "output/ademe_data_69_cleaned.csv",
    ROOT / "data/communes-france-2025.csv",
    ROOT / "data/climate_zones.csv",
]
missing_paths = [str(p.relative_to(ROOT)) for p in required_paths if not p.is_file()]
if missing_paths:
    raise FileNotFoundError(
        "Enrichissement Rhône non exécutable : fichiers absents : "
        + ", ".join(missing_paths)
        + ". Les modèles courants utilisent output/cleaned_data.csv, issu du notebook 1."
    )
print("Fichiers du périmètre Rhône disponibles.")


In [30]:
df = pd.read_csv(
    "output/ademe_data_69_cleaned.csv",
    low_memory=False
)

communes = pd.read_csv(
    "data/communes-france-2025.csv",
    low_memory=False
)

climate = pd.read_csv(
    "data/climate_zones.csv",
    low_memory=False
)

print("DPE :", df.shape)
print("Communes :", communes.shape)
print("Climat :", climate.shape)

DPE : (13481, 37)
Communes : (34935, 47)
Climat : (96, 2)


In [31]:
print("Dimensions :", df.shape)
print("\nColonnes :")
print(df.columns.tolist())

Dimensions : (13481, 37)

Colonnes :
['cout_total_5_usages', 'cout_chauffage', 'cout_eclairage', 'cout_refroidissement', 'cout_auxiliaires', 'cout_ecs', 'conso_5_usages_ef', 'conso_chauffage_ef', 'conso_eclairage_ef', 'conso_auxiliaires_ef', 'conso_ecs_ef', 'conso_refroidissement_ef', 'surface_habitable_logement', 'nombre_niveau_logement', 'type_batiment', 'annee_construction', 'code_insee_ban', 'code_departement_ban', 'etiquette_dpe', 'etiquette_ges', 'nom_commune_ban', 'code_postal_ban', 'emission_ges_chauffage', 'emission_ges_eclairage', 'emission_ges_ecs', 'emission_ges_5_usages', 'emission_ges_auxiliaires', 'emission_ges_refroidissement', 'type_energie_principale_chauffage', 'date_reception_dpe', 'numero_dpe', 'age_batiment', 'lat', 'lon', 'zone_climatique', 'code_insee', 'altitude_moyenne']


In [32]:
communes = pd.read_csv(
    "data/communes-france-2025.csv",
    low_memory=False
)

climate = pd.read_csv(
    "data/climate_zones.csv",
    low_memory=False
)

print("Communes :", communes.shape)
print("Zones climatiques :", climate.shape)

print("\nColonnes communes :")
print(communes.columns.tolist())

print("\nColonnes zones climatiques :")
print(climate.columns.tolist())

Communes : (34935, 47)
Zones climatiques : (96, 2)

Colonnes communes :
['Unnamed: 0', 'code_insee', 'nom_standard', 'nom_sans_pronom', 'nom_a', 'nom_de', 'nom_sans_accent', 'nom_standard_majuscule', 'typecom', 'typecom_texte', 'reg_code', 'reg_nom', 'dep_code', 'dep_nom', 'canton_code', 'canton_nom', 'epci_code', 'epci_nom', 'academie_code', 'academie_nom', 'code_postal', 'codes_postaux', 'zone_emploi', 'code_insee_centre_zone_emploi', 'code_unite_urbaine', 'nom_unite_urbaine', 'taille_unite_urbaine', 'type_commune_unite_urbaine', 'statut_commune_unite_urbaine', 'population', 'superficie_hectare', 'superficie_km2', 'densite', 'altitude_moyenne', 'altitude_minimale', 'altitude_maximale', 'latitude_mairie', 'longitude_mairie', 'latitude_centre', 'longitude_centre', 'grille_densite', 'grille_densite_texte', 'niveau_equipements_services', 'niveau_equipements_services_texte', 'gentile', 'url_wikipedia', 'url_villedereve']

Colonnes zones climatiques :
['Departement', 'Zone climatique']


In [33]:
print("Colonnes DPE :")
print(df.columns.tolist())

print("\nColonnes communes :")
print(communes.columns.tolist())

print("\nColonnes climat :")
print(climate.columns.tolist())

Colonnes DPE :
['cout_total_5_usages', 'cout_chauffage', 'cout_eclairage', 'cout_refroidissement', 'cout_auxiliaires', 'cout_ecs', 'conso_5_usages_ef', 'conso_chauffage_ef', 'conso_eclairage_ef', 'conso_auxiliaires_ef', 'conso_ecs_ef', 'conso_refroidissement_ef', 'surface_habitable_logement', 'nombre_niveau_logement', 'type_batiment', 'annee_construction', 'code_insee_ban', 'code_departement_ban', 'etiquette_dpe', 'etiquette_ges', 'nom_commune_ban', 'code_postal_ban', 'emission_ges_chauffage', 'emission_ges_eclairage', 'emission_ges_ecs', 'emission_ges_5_usages', 'emission_ges_auxiliaires', 'emission_ges_refroidissement', 'type_energie_principale_chauffage', 'date_reception_dpe', 'numero_dpe', 'age_batiment', 'lat', 'lon', 'zone_climatique', 'code_insee', 'altitude_moyenne']

Colonnes communes :
['Unnamed: 0', 'code_insee', 'nom_standard', 'nom_sans_pronom', 'nom_a', 'nom_de', 'nom_sans_accent', 'nom_standard_majuscule', 'typecom', 'typecom_texte', 'reg_code', 'reg_nom', 'dep_code', 'd

In [34]:
print("DPE :")
print(df[
    [
        "code_insee_ban",
        "code_departement_ban",
        "altitude_moyenne",
        "zone_climatique"
    ]
].head())

print("\nCommunes :")
print(
    communes[
        [
            "code_insee",
            "dep_code",
            "altitude_moyenne"
        ]
    ].head()
)

print("\nClimat :")
print(climate.head(10))

DPE :
   code_insee_ban  code_departement_ban  altitude_moyenne zone_climatique
0           69092                    69        225.000000              H1
1           69256                    69        174.000000              H1
2           69388                    69        383.932331              H1
3           69386                    69        383.932331              H1
4           69389                    69        383.932331              H1

Communes :
  code_insee dep_code  altitude_moyenne
0      01001       01               242
1      01002       01               483
2      01004       01               379
3      01005       01               290
4      01006       01               589

Climat :
  Departement Zone climatique
0          01             H1c
1          02             H1a
2          03             H1c
3          04             H2d
4          05             H1c
5          06              H3
6          07             H2d
7          08             H1b
8          09     

In [35]:
# ------------------------------------------------------------
# 1. Harmoniser les types des codes géographiques
# ------------------------------------------------------------

# Transformer le code INSEE des DPE en texte.
# zfill(5) garantit toujours 5 caractères :
# par exemple 1001 devient "01001".
df["code_insee_ban"] = (
    df["code_insee_ban"]
    .astype("Int64")
    .astype(str)
    .str.zfill(5)
)

# Transformer le code département des DPE en texte.
# zfill(2) transforme par exemple 1 en "01".
df["code_departement_ban"] = (
    df["code_departement_ban"]
    .astype("Int64")
    .astype(str)
    .str.zfill(2)
)

# Les codes du fichier communes sont aussi forcés en texte
# afin d'éviter les problèmes lors des jointures.
communes["code_insee"] = (
    communes["code_insee"]
    .astype(str)
    .str.zfill(5)
)

communes["dep_code"] = (
    communes["dep_code"]
    .astype(str)
    .str.zfill(2)
)

# Même traitement pour le fichier des zones climatiques.
climate["Departement"] = (
    climate["Departement"]
    .astype(str)
    .str.zfill(2)
)

In [36]:
# ------------------------------------------------------------
# 2. Vérifier que les formats sont maintenant compatibles
# ------------------------------------------------------------

print("DPE :")
print(
    df[
        ["code_insee_ban", "code_departement_ban"]
    ].head()
)

print("\nCommunes :")
print(
    communes[
        ["code_insee", "dep_code"]
    ].head()
)

print("\nClimat - département 69 :")
print(
    climate[
        climate["Departement"] == "69"
    ]
)

DPE :
  code_insee_ban code_departement_ban
0          69092                   69
1          69256                   69
2          69388                   69
3          69386                   69
4          69389                   69

Communes :
  code_insee dep_code
0      01001       01
1      01002       01
2      01004       01
3      01005       01
4      01006       01

Climat - département 69 :
   Departement Zone climatique
69          69             H1c


In [37]:
# ------------------------------------------------------------
# 7. Corriger certains codes INSEE pour la jointure
# ------------------------------------------------------------

# On crée une nouvelle colonne uniquement pour faire la jointure.
# Ainsi, on conserve le code INSEE original du DPE dans
# "code_insee_ban".
df["code_insee_jointure"] = df["code_insee_ban"].copy()


# ------------------------------------------------------------
# Cas 1 : les 9 arrondissements municipaux de Lyon
# ------------------------------------------------------------

# Les codes 69381 à 69389 correspondent aux arrondissements
# de Lyon.
# Le fichier des communes contient la commune de Lyon
# sous le code INSEE 69123.
codes_arrondissements_lyon = [
    "69381",
    "69382",
    "69383",
    "69384",
    "69385",
    "69386",
    "69387",
    "69388",
    "69389"
]

df.loc[
    df["code_insee_jointure"].isin(codes_arrondissements_lyon),
    "code_insee_jointure"
] = "69123"


# ------------------------------------------------------------
# Cas 2 : Pierre-Bénite
# ------------------------------------------------------------

# Depuis 2024, Pierre-Bénite (69152) fait partie de la
# commune nouvelle Oullins-Pierre-Bénite (69149).
df.loc[
    df["code_insee_jointure"] == "69152",
    "code_insee_jointure"
] = "69149"


# Vérification des modifications.
print(
    df[
        [
            "code_insee_ban",
            "code_insee_jointure",
            "nom_commune_ban"
        ]
    ]
    .drop_duplicates()
    .head(30)
)

   code_insee_ban code_insee_jointure              nom_commune_ban
0           69092               69092                       Gleizé
1           69256               69256               Vaulx-en-Velin
2           69388               69123                         Lyon
3           69386               69123                         Lyon
4           69389               69123                         Lyon
6           69387               69123                         Lyon
7           69264               69264  Villefranche-sur-Sa*u00f4ne
9           69285               69285                     Pusignan
12          69265               69265           Ville-sur-Jarnioux
13          69385               69123                         Lyon
14          69266               69266                 Villeurbanne
15          69264               69264       Villefranche-sur-Saône
16          69029               69029                         Bron
20          69381               69123                         

In [38]:
# ------------------------------------------------------------
# 8. Vérifier les codes INSEE utilisés pour les remplacements
# ------------------------------------------------------------

# On vérifie que les codes :
# - 69123 = Lyon
# - 69149 = Oullins-Pierre-Bénite
# existent bien dans le fichier des communes
# et disposent d'une altitude moyenne.

verification_codes = communes[
    communes["code_insee"].isin(
        ["69123", "69149"]
    )
][
    [
        "code_insee",
        "nom_standard",
        "altitude_moyenne"
    ]
]

print(verification_codes)

      code_insee           nom_standard  altitude_moyenne
27082      69123                   Lyon               237
27103      69149  Oullins-Pierre-Bénite               194


In [39]:
# ------------------------------------------------------------
# 9. Reconstruire proprement le dataset avant l'enrichissement
# ------------------------------------------------------------

# On repart du DataFrame DPE contenant maintenant
# "code_insee_jointure".
#
# On retire :
# - l'ancienne altitude_moyenne ;
# - l'ancienne zone_climatique ;
# - l'ancien code_insee issu de l'ancien enrichissement.
#
# L'objectif est de reconstruire ces informations
# uniquement à partir de nos fichiers sources.
df_enrichi = df.drop(
    columns=[
        "altitude_moyenne",
        "zone_climatique",
        "code_insee"
    ],
    errors="ignore"
).copy()

print("Dimensions avant nouvelle jointure :", df_enrichi.shape)

Dimensions avant nouvelle jointure : (13481, 35)


In [40]:
# ------------------------------------------------------------
# 10. Préparer la table altitude pour une jointure propre
# ------------------------------------------------------------

# On garde uniquement :
# - le code INSEE de la commune ;
# - son altitude moyenne.
communes_altitude = communes[
    [
        "code_insee",
        "altitude_moyenne"
    ]
].copy()

# On renomme "code_insee" pour avoir le même nom de clé
# que dans notre dataset DPE.
communes_altitude = communes_altitude.rename(
    columns={
        "code_insee": "code_insee_jointure"
    }
)

print(communes_altitude.head())

  code_insee_jointure  altitude_moyenne
0               01001               242
1               01002               483
2               01004               379
3               01005               290
4               01006               589


In [41]:
# ------------------------------------------------------------
# 11. Ajouter l'altitude moyenne avec les codes corrigés
# ------------------------------------------------------------

# On réalise la jointure entre :
# - df_enrichi : notre dataset DPE
# - communes_altitude : la table des communes avec leur altitude
#
# La clé commune est "code_insee_jointure".
#
# how="left" permet de conserver tous les DPE,
# même si certaines communes n'ont pas de correspondance.
#
# validate="many_to_one" vérifie qu'un même code INSEE
# dans la table des communes ne correspond qu'à une seule altitude.
df_enrichi = df_enrichi.merge(
    communes_altitude,
    how="left",
    on="code_insee_jointure",
    validate="many_to_one"
)

# Vérifier que le nombre de lignes n'a pas changé.
print(
    "Dimensions après nouvelle jointure :",
    df_enrichi.shape
)

# Vérifier combien de logements n'ont toujours pas d'altitude.
print(
    "Altitudes manquantes après correction :",
    df_enrichi["altitude_moyenne"].isna().sum()
)

Dimensions après nouvelle jointure : (13481, 36)
Altitudes manquantes après correction : 0


In [42]:
# ------------------------------------------------------------
# 12. Préparer la table des zones climatiques
# ------------------------------------------------------------

# On garde uniquement les deux colonnes utiles :
# - Departement : clé de jointure
# - Zone climatique : information à ajouter au dataset DPE
climate_clean = climate[
    [
        "Departement",
        "Zone climatique"
    ]
].copy()

# On renomme les colonnes pour avoir des noms plus simples
# et cohérents avec notre dataset.
climate_clean = climate_clean.rename(
    columns={
        "Departement": "code_departement_ban",
        "Zone climatique": "zone_climatique"
    }
)

# Vérification rapide.
print(climate_clean.head())

# Vérifier la zone climatique du Rhône.
print(
    "\nZone climatique du Rhône :"
)
print(
    climate_clean[
        climate_clean["code_departement_ban"] == "69"
    ]
)

  code_departement_ban zone_climatique
0                   01             H1c
1                   02             H1a
2                   03             H1c
3                   04             H2d
4                   05             H1c

Zone climatique du Rhône :
   code_departement_ban zone_climatique
69                   69             H1c


In [43]:
# ------------------------------------------------------------
# 13. Ajouter la zone climatique aux données DPE
# ------------------------------------------------------------

# On relie le dataset DPE enrichi avec la table des zones climatiques
# grâce au code du département.
#
# how="left" permet de conserver tous les DPE,
# même si une correspondance climatique n'était pas trouvée.
#
# validate="many_to_one" vérifie qu'un département
# ne correspond qu'à une seule zone climatique.
df_enrichi = df_enrichi.merge(
    climate_clean,
    how="left",
    on="code_departement_ban",
    validate="many_to_one"
)

# Vérifier les dimensions après l'ajout de la zone climatique.
print(
    "Dimensions après ajout du climat :",
    df_enrichi.shape
)

# Vérifier s'il reste des zones climatiques manquantes.
print(
    "Zones climatiques manquantes :",
    df_enrichi["zone_climatique"].isna().sum()
)

# Voir les différentes zones climatiques présentes
# et leur nombre d'occurrences.
print("\nZones climatiques présentes :")
print(
    df_enrichi["zone_climatique"].value_counts(dropna=False)
)

Dimensions après ajout du climat : (13481, 37)
Zones climatiques manquantes : 0

Zones climatiques présentes :
zone_climatique
H1c    13481
Name: count, dtype: int64


In [44]:
# ------------------------------------------------------------
# 14. Contrôles finaux de l'enrichissement
# ------------------------------------------------------------

# Vérifier qu'on n'a pas créé de doublons supplémentaires.
print(
    "Nombre de lignes dupliquées :",
    df_enrichi.duplicated().sum()
)

# Vérifier les valeurs manquantes sur les colonnes enrichies.
print("\nValeurs manquantes :")
print(
    df_enrichi[
        [
            "altitude_moyenne",
            "zone_climatique"
        ]
    ].isna().sum()
)

# Vérifier quelques lignes du dataset final.
print("\nExemple de données enrichies :")
print(
    df_enrichi[
        [
            "code_insee_ban",
            "nom_commune_ban",
            "altitude_moyenne",
            "zone_climatique"
        ]
    ].head(10)
)

Nombre de lignes dupliquées : 0

Valeurs manquantes :
altitude_moyenne    0
zone_climatique     0
dtype: int64

Exemple de données enrichies :
  code_insee_ban              nom_commune_ban  altitude_moyenne  \
0          69092                       Gleizé               225   
1          69256               Vaulx-en-Velin               174   
2          69388                         Lyon               237   
3          69386                         Lyon               237   
4          69389                         Lyon               237   
5          69386                         Lyon               237   
6          69387                         Lyon               237   
7          69264  Villefranche-sur-Sa*u00f4ne               181   
8          69388                         Lyon               237   
9          69285                     Pusignan               219   

  zone_climatique  
0             H1c  
1             H1c  
2             H1c  
3             H1c  
4             H1c  

In [45]:
# ------------------------------------------------------------
# 15. Vérifier les variables nécessaires aux modèles ML
# ------------------------------------------------------------

# Liste des variables utilisées par les modèles
# de classification et de régression.
colonnes_modeles = [
    "surface_habitable_logement",
    "nombre_niveau_logement",
    "age_batiment",
    "altitude_moyenne",
    "type_energie_principale_chauffage",
    "type_batiment",
    "zone_climatique",
    "cout_total_5_usages",
    "conso_5_usages_ef",
    "etiquette_dpe"
]

# Chercher les colonnes qui seraient absentes
# du dataset enrichi.
colonnes_manquantes = [
    colonne
    for colonne in colonnes_modeles
    if colonne not in df_enrichi.columns
]

print(
    "Colonnes manquantes pour les modèles :",
    colonnes_manquantes
)

Colonnes manquantes pour les modèles : []


In [46]:
# ------------------------------------------------------------
# 16. Sauvegarder le dataset enrichi final
# ------------------------------------------------------------

# Le dataset final contient maintenant :
# - les données DPE nettoyées ;
# - l'altitude moyenne ;
# - la zone climatique corrigée.
df_enrichi.to_csv(
    "output/ademe_data_69_enriched.csv",
    index=False
)

print("Dataset enrichi sauvegardé.")

Dataset enrichi sauvegardé.


In [47]:
# ------------------------------------------------------------
# 17. Vérifier que le fichier final existe
# ------------------------------------------------------------

fichier_final = "output/ademe_data_69_enriched.csv"

print(
    "Fichier créé :",
    os.path.exists(fichier_final)
)

print(
    "Dimensions finales :",
    df_enrichi.shape
)

Fichier créé : True
Dimensions finales : (13481, 37)
